In [23]:
import sqlite3
import pandas as pd

# 1. Connect to your local database file
conn = sqlite3.connect("app_analytics.db")

# 2. Store your SQL query as a multi-line Python string
query = """
SELECT event_name, COUNT(1) AS event_count
FROM staging_app_events
GROUP BY event_name
ORDER BY event_count DESC;
"""

# 3. Use Pandas to run the SQL query and automatically format it as a clean dataframe
df_results = pd.read_sql_query(query, conn)

# 4. Close the database connection
conn.close()

# 5. Display the beautiful output table
df_results

,event_name,event_count
0,app_open,15299
1,view_task_list,12889
2,start_task,10781
3,complete_task,7036
4,claim_points,3693
5,server_error_500,970
6,ap_open,616
7,view_tasklist,560
8,clck_reward,158


In [25]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("app_analytics.db")
cursor = conn.cursor()

print("Creating the clean_app_events view...")
# SQL Script to create a cleaned view handling event standardization
cursor.execute("""
CREATE VIEW IF NOT EXISTS clean_app_events AS
SELECT 
    timestamp,
    user_id,
    session_id,
    device_os,
    CASE 
        WHEN event_name = 'ap_open' THEN 'app_open'
        WHEN event_name = 'view_tasklist' THEN 'view_task_list'
        WHEN event_name = 'clck_reward' THEN 'claim_points'
        ELSE event_name 
    END AS clean_event_name
FROM staging_app_events;
""")
conn.commit()

print("Verifying cleaned event names...")
# Let's run a query against our NEW view to check if it worked
verification_query = """
SELECT clean_event_name, COUNT(1) AS event_count
FROM clean_app_events
GROUP BY clean_event_name
ORDER BY event_count DESC;
"""

df_clean_summary = pd.read_sql_query(verification_query, conn)
conn.close()

# Show the results
df_clean_summary

Creating the clean_app_events view...
Verifying cleaned event names...


,clean_event_name,event_count
0,app_open,15915
1,view_task_list,13449
2,start_task,10781
3,complete_task,7036
4,claim_points,3851
5,server_error_500,970


In [27]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("app_analytics.db")
cursor = conn.cursor()

print("Dropping old view...")
cursor.execute("DROP VIEW IF EXISTS clean_app_events;")

print("Creating the master fully-cleaned view...")
# This query handles Typos, Unix Timestamps, and Deduplication all at once!
cursor.execute("""
CREATE VIEW clean_app_events AS
SELECT DISTINCT
    CASE 
        WHEN timestamp LIKE '%-%' THEN timestamp
        ELSE datetime(CAST(timestamp AS INT), 'unixepoch')
    END AS clean_timestamp,
    user_id,
    session_id,
    device_os,
    CASE 
        WHEN event_name = 'ap_open' THEN 'app_open'
        WHEN event_name = 'view_tasklist' THEN 'view_task_list'
        WHEN event_name = 'clck_reward' THEN 'claim_points'
        ELSE event_name 
    END AS clean_event_name
FROM staging_app_events;
""")
conn.commit()

print("Pipeline updated! Let's check the new row count to see if duplicates were removed...")
# Count the rows in our clean view vs the raw staging table
df_counts = pd.read_sql_query("""
    SELECT 
        (SELECT COUNT(1) FROM staging_app_events) AS raw_row_count,
        (SELECT COUNT(1) FROM clean_app_events) AS clean_row_count
""", conn)

conn.close()
df_counts

Dropping old view...
Creating the master fully-cleaned view...
Pipeline updated! Let's check the new row count to see if duplicates were removed...


,raw_row_count,clean_row_count
0,52002,51383


In [29]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("app_analytics.db")

# SQL Query using conditional aggregation to trace user counts per step
funnel_query = """
SELECT 
    device_os,
    COUNT(DISTINCT CASE WHEN clean_event_name = 'app_open' THEN user_id END) AS step_1_open,
    COUNT(DISTINCT CASE WHEN clean_event_name = 'view_task_list' THEN user_id END) AS step_2_view,
    COUNT(DISTINCT CASE WHEN clean_event_name = 'start_task' THEN user_id END) AS step_3_start,
    COUNT(DISTINCT CASE WHEN clean_event_name = 'complete_task' THEN user_id END) AS step_4_complete,
    COUNT(DISTINCT CASE WHEN clean_event_name = 'claim_points' THEN user_id END) AS step_5_claim
FROM clean_app_events
GROUP BY device_os;
"""

df_funnel = pd.read_sql_query(funnel_query, conn)
conn.close()

# Display the funnel numbers
df_funnel

,device_os,step_1_open,step_2_view,step_3_start,step_4_complete,step_5_claim
0,Android,1519,1517,1500,1435,1093
1,iOS,976,973,965,930,835


In [31]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("app_analytics.db")

# SQL Query to isolate where the server errors are happening
error_query = """
SELECT 
    device_os, 
    clean_event_name, 
    COUNT(1) AS total_errors
FROM clean_app_events
WHERE clean_event_name = 'server_error_500'
GROUP BY device_os;
"""

df_errors = pd.read_sql_query(error_query, conn)
conn.close()

df_errors

,device_os,clean_event_name,total_errors
0,Android,server_error_500,970
